# 04 — Météo NASA POWER
Données journalières + climatologie ; indicateurs (cumuls, périodes sèches, anomalies). Estimations maillées, pas des mesures sur parcelle.

In [ ]:
# --- Bootstrap (Kaggle or local) -------------------------------------------------
# Kaggle: Settings > Internet = ON ; Add-ons > Secrets > add "MODEL_API_KEY" (and "GITHUB_TOKEN" if the repo is private)
import os, subprocess, sys
REPO = "code-htech07/sina-meta"
BRANCH = "ccr-c8fab4b3-0jmvi1"  # change to "main" once merged
if os.path.exists("engine"):            # already inside the repo (local run)
    pass
else:
    if not os.path.exists("sina-meta"):
        url = f"https://github.com/{REPO}.git"
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
            url = f"https://{tok}@github.com/{REPO}.git"
        except Exception:
            pass
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "sina-meta"], check=True,
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # silent: never print a tokenised URL
    os.chdir("sina-meta")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, os.getcwd())
from engine.config import describe_secret_status
from engine.logging_utils import setup_logging
setup_logging("WARNING")
print(describe_secret_status())  # shows only presence/length of the key, never its value

In [ ]:
from datetime import date
import pandas as pd
from engine.sources.weather.nasa_power import get_weather_context
w = get_weather_context(5.70, -6.70, date.today())
print(w.status, w.limitations)
print(w.indicators.model_dump_json(indent=1))
df = pd.DataFrame([d.model_dump() for d in w.daily]).set_index('day')
df[['precipitation_mm']].plot.bar(figsize=(14,3), legend=False, title='Pluie journalière (mm)');